In [245]:
import pandas as pd
import numpy as np

In [246]:
df = pd.read_csv("dataset_area/copernicus_2016_2025.csv", low_memory=False)

print(df.head())

            valid_time  latitude  longitude       u10       v10        d2m  \
0  2016-07-01 00:00:00     41.05      -8.68  1.485055 -2.932715  289.00098   
1  2016-07-01 01:00:00     41.05      -8.68  1.378376 -3.178485  288.86182   
2  2016-07-01 02:00:00     41.05      -8.68  1.137827 -3.516484  288.59668   
3  2016-07-01 03:00:00     41.05      -8.68  0.749619 -3.548216  288.49194   
4  2016-07-01 04:00:00     41.05      -8.68  0.577462 -3.476008  288.45264   

         t2m         sp        skt      u10n  ...  hcc       lcc  mcc  \
0  289.73047  101106.31  290.39478  1.533758  ...  0.0  0.995655  0.0   
1  289.82910  101108.06  290.52660  1.444549  ...  0.0  1.000000  0.0   
2  289.58570  101062.00  290.38990  1.195093  ...  0.0  1.000000  0.0   
3  289.42944  101038.50  290.29346  0.788165  ...  0.0  1.000000  0.0   
4  289.35718  101033.56  290.23560  0.599234  ...  0.0  1.000000  0.0   

        tcc        blh      cape  number  expver  origem_ano  origem_pasta  
0  0.995655  25

In [247]:
# Drop column
df.drop('number', axis=1, inplace=True)
df.drop('expver', axis=1, inplace=True)
df.drop('origem_ano', axis=1, inplace=True)
df.drop('origem_pasta', axis=1, inplace=True)

In [248]:
# Convert to Celsius
df['temp_celsius'] = df['t2m'] - 273.15
df['td'] = df['d2m'] - 273.15

In [249]:
# humidity
df['relative_humidity'] = 100 * np.exp(((17.67 * df['td'])/(df['td'] + 243.5)) - ((17.67 * df['temp_celsius'])/(df['temp_celsius'] + 243.5)))

# Calculate wind speed
df['wind_speed'] = np.sqrt(df['u10']**2 + df['v10']**2)

In [250]:
# Temperature
# Thermal gradient at surface
df['gradient'] = df['skt'] - df['t2m']

In [251]:
# enosr concentrations
df['so_concentration'] = (np.abs(df['u10']) * df['wind_speed'] *(df['relative_humidity']/100))

In [252]:
df.drop('latitude', axis=1, inplace=True)
df.drop('longitude', axis=1, inplace=True)
#df.drop('tp', axis=1, inplace=True)
df.drop('skt', axis=1, inplace=True)
df.drop('u10', axis=1, inplace=True)
df.drop('v10', axis=1, inplace=True)
#df.drop('lsp', axis=1, inplace=True)
#df.drop('ssr', axis=1, inplace=True)
#df.drop('ssrc', axis=1, inplace=True)
#df.drop('ssrd', axis=1, inplace=True)
#df.drop('ssrdc', axis=1, inplace=True)
#df.drop('str', axis=1, inplace=True)
#df.drop('strc', axis=1, inplace=True)
#df.drop('strd', axis=1, inplace=True)
#df.drop('strdc', axis=1, inplace=True)
#df.drop('uvb', axis=1, inplace=True)
df.drop('aluvd', axis=1, inplace=True)
df.drop('aluvp', axis=1, inplace=True)
df.drop('u10n', axis=1, inplace=True)
df.drop('v10n', axis=1, inplace=True)
#df.drop('fg10', axis=1, inplace=True)
df.drop('i10fg', axis=1, inplace=True)
df.drop('t2m', axis=1, inplace=True)
df.drop('d2m', axis=1, inplace=True)
#df.drop('mx2t', axis=1, inplace=True)
#df.drop('mn2t', axis=1, inplace=True)
df.drop('sp', axis=1, inplace=True)
df.drop('cbh', axis=1, inplace=True)
df.drop('lcc', axis=1, inplace=True)
df.drop('mcc', axis=1, inplace=True)
df.drop('hcc', axis=1, inplace=True)
df.drop('tcc', axis=1, inplace=True)
df.drop('blh', axis=1, inplace=True)
df.drop('cape', axis=1, inplace=True)
#df.drop('slhf', axis=1, inplace=True)
#df.drop('cp', axis=1, inplace=True)
#df.drop('cdir', axis=1, inplace=True)
#df.drop('fdir', axis=1, inplace=True)
#df.drop('t', axis=1, inplace=True)
df.drop('td', axis=1, inplace=True)
#df.drop('ssrd_hourly', axis=1, inplace=True)
df.drop('wind_speed', axis=1, inplace=True)
df.drop('gradient', axis=1, inplace=True)
#df.drop('fdir_hourly', axis=1, inplace=True)
#df.drop('uv_temp_index', axis=1, inplace=True)
#df.drop('clear_sky_ratio', axis=1, inplace=True)
#df.drop('rain_hourly', axis=1, inplace=True)
#df.drop('rain_intensity', axis=1, inplace=True)

In [253]:
# Factors
# water
condictions_factor = [
    (df['relative_humidity'] < 60),
    (df['relative_humidity'] >= 60) & (df['relative_humidity'] < 75),
    (df['relative_humidity'] >= 75) & (df['relative_humidity'] <= 90),
    (df['relative_humidity'] > 90)
]

fatores = [0, 0.33, 0.66, 1]

df['Fhu'] = np.select(condictions_factor, fatores)

# enosr
cond_so2 = [(df['so_concentration'] < 5),
    (df['so_concentration'] >= 5) & (df['so_concentration'] < 30),
    (df['so_concentration'] >= 30) & (df['so_concentration'] <= 90),
    (df['so_concentration'] >= 90) & (df['so_concentration'] <= 250)
]

fatores_so2 = [0.2, 0.5, 0.8, 1]
df['Fso'] = np.select(cond_so2, fatores_so2)

#temperatura
cond_temp = [(df['temp_celsius'] >= 10) & (df['temp_celsius'] < 20),
    (df['temp_celsius'] >= 20) & (df['temp_celsius'] <= 30),
    (df['temp_celsius'] >= 30) & (df['temp_celsius'] <= 45),
    (df['temp_celsius'] >= 45) & (df['temp_celsius'] <= 60),
    (df['temp_celsius'] >= 60)
]

fatores_temp = [0, 0.2, 0.33, 0.66, 1]

df['Ftemp'] = np.select(cond_temp, fatores_temp)

# calculate value of agressive index
df['Index_agressive'] = (df['Fhu'] * 0.2) + (df['Fso'] * 0.1) + (df['Ftemp'] * 0.15)

# Calculate Estimated Live
useful_live = 25 # can be change

df['ESL'] = useful_live - useful_live * df['Fhu'] * df['Fso'] * df['Ftemp']

In [254]:
# Grouping
df['valid_time'] = pd.to_datetime(df['valid_time'])
df['tow'] = ((df['relative_humidity'] > 80) & (df['temp_celsius'] > 0)).astype(int)
df['wet_state'] = (df['relative_humidity'] > 80).astype(int)
df['wet_dry_cycle'] = (df['wet_state'].diff().abs() == 1).astype(int)

df_6m = df.groupby(pd.Grouper(key='valid_time', freq='6MS')).agg({
    'relative_humidity': ['mean', 'max', 'std'],
    'temp_celsius': ['mean', 'max', 'min'],
    'so_concentration': ['mean', 'max'],
    'Fhu': 'mean',
    'Fso': 'mean',
    'Ftemp': 'mean',
    'ESL': 'mean',
    'Index_agressive': ['mean', 'max'],
    'tow': ['sum', 'mean'],
    'wet_dry_cycle': 'sum'
}).reset_index()

df_6m.columns = ['_'.join(col) if isinstance(col, tuple) else col for col in df_6m.columns]

df_6m['cum_rh'] = df_6m['relative_humidity_mean'].cumsum()
df_6m['cum_so'] = df_6m['so_concentration_mean'].cumsum()
df_6m['cum_aggressive'] = df_6m['Index_agressive_mean'].cumsum()

print(df_6m.head())

  valid_time_  relative_humidity_mean  relative_humidity_max  \
0  2016-07-01               77.121939              97.701318   
1  2017-01-01               76.689790              98.111909   
2  2017-07-01               77.422855              99.476511   
3  2018-01-01               80.169744              98.498260   
4  2018-07-01               79.174887              99.252217   

   relative_humidity_std  temp_celsius_mean  temp_celsius_max  \
0              13.502818          17.481987          32.71523   
1              12.947081          14.673092          29.61930   
2              13.596748          16.848321          31.21940   
3              11.287664          13.164400          27.82046   
4              13.459271          17.454106          32.68716   

   temp_celsius_min  so_concentration_mean  so_concentration_max  Fhu_mean  \
0           5.39028               6.024837             82.214778  0.551144   
1           2.56924               8.340291            155.483575  0.

In [255]:
# Save csv
df_6m.to_csv('dataset_area/copernicus_ds.csv', index=False)
#df_6m.to_csv('copernicus_ds_test.csv', index=False)

In [256]:
#df_damage = pd.read_csv("Dataset_image_test.csv")
#df_weather = pd.read_csv("copernicus_ds_test.csv")

df_damage = pd.read_csv("dataset_area/Image_Dataset.csv")
df_weather = pd.read_csv("dataset_area/copernicus_ds.csv")

df_damage['data'] = pd.to_datetime(df_damage['data'])

df_weather['valid_time_'] = pd.to_datetime(df_weather['valid_time_'])

df_damage['data_ref'] = df_damage['data'].dt.date
df_weather['data_ref'] = df_weather['valid_time_'].dt.date

df_damage = df_damage[df_damage['AnomalyType'] == 'corrosion'].copy()

df_final = pd.merge(
    df_damage,
    df_weather,
    on='data_ref',
    how='left'
)

# Date
df_final['data'] = pd.to_datetime(df_final['data'])
df_final['first_date'] = (df_final.groupby('AnomalyID')['data'].transform('min'))
df_final['age_days'] = (df_final['data'] - df_final['first_date']).dt.days

df_final = df_final.sort_values(['AnomalyID', 'data'])

df_final['prev_area'] = (df_final.groupby('AnomalyID')['Area (cm2)'].shift(1)).fillna(0)
df_final['prev_delta'] = (df_final.groupby('AnomalyID')['Delta_area'].shift(1)).fillna(0)

df_final['is_growth'] = (df_final['Delta_area'] > 0).astype(int)

cols_to_drop = [
    'Coordinates', 'Coordinates (x,y)', 'ImagePath', 'tempo_t',
    'anos_degradacao', 'meses_degradacao', 'data_referencia',
    'arquivo_origem', 'data_ref', 'valid_time', 'Faixa_Area',
    'ID_Figura', 'first_date', 'Figura_name_encoded', 'FacadeRoofID_encoded',
    'Delta_area_lag1', 'SimulationDate', 'SurfaceGrowthMethod', 'Figure_name'
]

df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns], inplace=True)
df_final = df_final.drop_duplicates(subset=['data', 'AnomalyID']).reset_index(drop=True)

df_final.to_csv("dataset_area/final.csv", index=False)

print("União concluída!")
print(f"Total de linhas no dataset final: {len(df_final)}")
print(df_final.head())

União concluída!
Total de linhas no dataset final: 40402
        data   AnomalyID FacadeRoofID AnomalyType  Area (cm2)  \
0 2016-07-01  Figura_1#0           FN   corrosion      4.7349   
1 2017-01-01  Figura_1#0           FN   corrosion      4.8075   
2 2017-07-01  Figura_1#0           FN   corrosion      4.8801   
3 2018-01-01  Figura_1#0           FN   corrosion      4.9346   
4 2018-07-01  Figura_1#0           FN   corrosion      5.0253   

   Surface3DArea_cm2  Added3DArea_cm2  SurfaceVertexCount  SurfaceNormalX  \
0             4.7349           0.1089                20.0       -0.224089   
1             4.8075           0.0726                24.0       -0.219070   
2             4.8801           0.0726                28.0       -0.210268   
3             4.9346           0.0545                31.0       -0.204188   
4             5.0253           0.0908                36.0       -0.204407   

   SurfaceNormalY  ...  tow_sum  tow_mean  wet_dry_cycle_sum      cum_rh  \
0        0.93

For Data test

In [257]:
df_test = pd.read_csv("dataset_area/dados_2026_Jan_Jun.csv", low_memory=False)

print(df_test)

               valid_time  latitude  longitude       u10       v10        d2m  \
0     2026-01-01 00:00:00     41.05      -8.68 -4.818805  1.466759  277.93190   
1     2026-01-01 01:00:00     41.05      -8.68 -5.034215  1.124702  277.90698   
2     2026-01-01 02:00:00     41.05      -8.68 -4.813668  1.203621  277.81030   
3     2026-01-01 03:00:00     41.05      -8.68 -4.521911  1.526177  278.00684   
4     2026-01-01 04:00:00     41.05      -8.68 -5.055226  1.600964  278.11133   
...                   ...       ...        ...       ...       ...        ...   
1068  2026-02-14 12:00:00     41.05      -8.68  1.851047 -6.220078  279.38403   
1069  2026-02-14 13:00:00     41.05      -8.68  2.854895 -6.313930  279.26270   
1070  2026-02-14 14:00:00     41.05      -8.68  3.084322 -6.779871  279.37476   
1071  2026-02-14 15:00:00     41.05      -8.68  2.650987 -6.226871  279.08936   
1072  2026-02-14 16:00:00     41.05      -8.68  2.764143 -5.752665  279.35840   

            t2m          sp

In [258]:
# Drop column
df_test.drop('number', axis=1, inplace=True)
df_test.drop('expver', axis=1, inplace=True)

In [259]:
# Convert to Celsius
df_test['temp_celsius'] = df_test['t2m'] - 273.15
df_test['td'] = df_test['d2m'] - 273.15

In [260]:
# humidity
df_test['relative_humidity'] = 100 * np.exp(((17.67 * df_test['td'])/(df_test['td'] + 243.5)) - ((17.67 * df_test['temp_celsius'])/(df_test['temp_celsius'] + 243.5)))

# Calculate wind speed
df_test['wind_speed'] = np.sqrt(df_test['u10']**2 + df_test['v10']**2)

In [261]:
# Temperature
# Thermal gradient at surface
df_test['gradient'] = df_test['skt'] - df_test['t2m']
# enosr concentrations
df_test['so_concentration'] = (np.abs(df_test['u10']) * df_test['wind_speed'] *(df_test['relative_humidity']/100))

In [262]:
df_test.drop('latitude', axis=1, inplace=True)
df_test.drop('longitude', axis=1, inplace=True)
df_test.drop('skt', axis=1, inplace=True)
df_test.drop('u10', axis=1, inplace=True)
df_test.drop('v10', axis=1, inplace=True)
df_test.drop('aluvd', axis=1, inplace=True)
df_test.drop('aluvp', axis=1, inplace=True)
df_test.drop('u10n', axis=1, inplace=True)
df_test.drop('v10n', axis=1, inplace=True)
df_test.drop('i10fg', axis=1, inplace=True)
df_test.drop('t2m', axis=1, inplace=True)
df_test.drop('d2m', axis=1, inplace=True)
df_test.drop('sp', axis=1, inplace=True)
df_test.drop('cbh', axis=1, inplace=True)
df_test.drop('lcc', axis=1, inplace=True)
df_test.drop('mcc', axis=1, inplace=True)
df_test.drop('hcc', axis=1, inplace=True)
df_test.drop('tcc', axis=1, inplace=True)
df_test.drop('blh', axis=1, inplace=True)
df_test.drop('cape', axis=1, inplace=True)
df_test.drop('td', axis=1, inplace=True)
df_test.drop('wind_speed', axis=1, inplace=True)
df_test.drop('gradient', axis=1, inplace=True)

In [263]:
# Factors
# water
condictions_factor = [
    (df_test['relative_humidity'] < 60),
    (df_test['relative_humidity'] >= 60) & (df_test['relative_humidity'] < 75),
    (df_test['relative_humidity'] >= 75) & (df_test['relative_humidity'] <= 90),
    (df_test['relative_humidity'] > 90)
]

fatores = [0, 0.33, 0.66, 1]

df_test['Fhu'] = np.select(condictions_factor, fatores)

# enosr
cond_so2 = [(df_test['so_concentration'] < 5),
    (df_test['so_concentration'] >= 5) & (df_test['so_concentration'] < 30),
    (df_test['so_concentration'] >= 30) & (df_test['so_concentration'] <= 90),
    (df_test['so_concentration'] >= 90) & (df_test['so_concentration'] <= 250)
]

fatores_so2 = [0.2, 0.5, 0.8, 1]
df_test['Fso'] = np.select(cond_so2, fatores_so2)

#temperatura
cond_temp = [(df_test['temp_celsius'] >= 10) & (df_test['temp_celsius'] < 20),
    (df_test['temp_celsius'] >= 20) & (df_test['temp_celsius'] <= 30),
    (df_test['temp_celsius'] >= 30) & (df_test['temp_celsius'] <= 45),
    (df_test['temp_celsius'] >= 45) & (df_test['temp_celsius'] <= 60),
    (df_test['temp_celsius'] >= 60)
]

fatores_temp = [0, 0.2, 0.33, 0.66, 1]

df_test['Ftemp'] = np.select(cond_temp, fatores_temp)

# calculate value of agressive index
df_test['Index_agressive'] = (df_test['Fhu'] * 0.2) + (df_test['Fso'] * 0.1) + (df_test['Ftemp'] * 0.15)

# Calculate Estimated Live
useful_live = 25 # can be change

df_test['ESL'] = useful_live - useful_live * df_test['Fhu'] * df_test['Fso'] * df_test['Ftemp']

In [264]:
# Grouping
df_test['valid_time'] = pd.to_datetime(df_test['valid_time'])
df_test['tow'] = ((df_test['relative_humidity'] > 80) & (df_test['temp_celsius'] > 0)).astype(int)
df_test['wet_state'] = (df_test['relative_humidity'] > 80).astype(int)
df_test['wet_dry_cycle'] = (df_test['wet_state'].diff().abs() == 1).astype(int)

df_6m_test = df_test.groupby(pd.Grouper(key='valid_time', freq='6MS')).agg({
    'relative_humidity': ['mean', 'max', 'std'],
    'temp_celsius': ['mean', 'max', 'min'],
    'so_concentration': ['mean', 'max'],
    'Fhu': 'mean',
    'Fso': 'mean',
    'Ftemp': 'mean',
    'ESL': 'mean',
    'Index_agressive': ['mean', 'max'],
    'tow': ['sum', 'mean'],
    'wet_dry_cycle': 'sum'
}).reset_index()

df_6m_test.columns = ['_'.join(col) if isinstance(col, tuple) else col for col in df_6m_test.columns]

df_6m_test['cum_rh'] = df_6m_test['relative_humidity_mean'].cumsum()
df_6m_test['cum_so'] = df_6m_test['so_concentration_mean'].cumsum()
df_6m_test['cum_aggressive'] = df_6m_test['Index_agressive_mean'].cumsum()

df_6m_test.to_csv('dataset_area/copernicus_ds_test.csv', index=False)

In [265]:
# 1. Carregar o histórico de 2025 e os novos dados de 2026
df_history_2025 = pd.read_csv("dataset_area/final.csv")
df_damage_test = pd.read_csv("dataset_area/Image_Dataset_Test.csv")
df_weather_test = pd.read_csv("dataset_area/copernicus_ds_test.csv")

# 2. Processar datas e referências
df_damage_test['data'] = pd.to_datetime(df_damage_test['data'])
df_weather_test['valid_time_'] = pd.to_datetime(df_weather_test['valid_time_'])

df_damage_test['data_ref'] = df_damage_test['data'].dt.date
df_weather_test['data_ref'] = df_weather_test['valid_time_'].dt.date

df_damage_test = df_damage_test[df_damage_test['AnomalyType'] == 'corrosion'].copy()

df_new_2026 = pd.merge(
    df_damage_test,
    df_weather_test,
    on='data_ref',
    how='left'
)

# 3. Juntar o histórico de 2025 com os dados novos de 2026
df_combined = pd.concat([df_history_2025, df_new_2026], ignore_index=True)

# 4. Converter a data para datetime e ordenar rigorosamente por anomalia e data
df_combined['data'] = pd.to_datetime(df_combined['data'])
df_combined = df_combined.sort_values(['AnomalyID', 'data']).drop_duplicates(subset=['data', 'AnomalyID'])

# 5. Calcular a idade e a diferença de área entre semestres consecutivos
df_combined['first_date'] = df_combined.groupby('AnomalyID')['data'].transform('min')
df_combined['age_days'] = (df_combined['data'] - df_combined['first_date']).dt.days

# AQUI É FEITA A DIFERENÇA ENTRE SEMESTRES (subtrai o valor da linha anterior do mesmo AnomalyID)
df_combined['Delta_area'] = df_combined.groupby('AnomalyID')['Area (cm2)'].diff().fillna(0)

df_combined['prev_area'] = df_combined.groupby('AnomalyID')['Area (cm2)'].shift(1).fillna(0)
df_combined['prev_delta'] = df_combined.groupby('AnomalyID')['Delta_area'].shift(1).fillna(0)
df_combined['is_growth'] = (df_combined['Delta_area'] > 0).astype(int)

# 6. Filtrar APENAS para o ano de 2026 (mantendo já o Delta_area calculado com base no final de 2025)
df_final_test = df_combined[df_combined['data'].dt.year == 2026].copy()

# 7. Limpar colunas desnecessárias
cols_to_drop = [
    'Coordinates', 'Coordinates (x,y)', 'ImagePath', 'tempo_t',
    'anos_degradacao', 'meses_degradacao', 'data_referencia',
    'arquivo_origem', 'data_ref', 'valid_time', 'Faixa_Area',
    'ID_Figura', 'first_date', 'Figura_name_encoded', 'FacadeRoofID_encoded',
    'Delta_area_lag1', 'SimulationDate', 'SurfaceGrowthMethod', 'Figure_name'
]

df_final_test.drop(columns=[c for c in cols_to_drop if c in df_final_test.columns], inplace=True)
df_final_test = df_final_test.reset_index(drop=True)

# 8. Guardar o resultado final
df_final_test.to_csv("dataset_area/final_test.csv", index=False)

print("Cálculo da diferença entre semestres concluído!")
print(f"Total de linhas no dataset final (2026): {len(df_final_test)}")
print(df_final_test[['data', 'AnomalyID', 'Area (cm2)', 'Delta_area', 'prev_area']].head())

Cálculo da diferença entre semestres concluído!
Total de linhas no dataset final (2026): 2150
        data     AnomalyID  Area (cm2)  Delta_area  prev_area
0 2026-01-01    Figura_1#0      5.9980      0.0544     5.9436
1 2026-01-01    Figura_1#1      8.2419      0.0000     8.2419
2 2026-01-01  Figura_100#1    165.2811      0.0000   165.2811
3 2026-01-01  Figura_100#3      1.1981      0.0000     1.1981
4 2026-01-01  Figura_100#4    242.8175      2.5490   240.2685
